# Alex's Walk Home: Parameter Exploration

This notebook explores how different probability values for **p_kaia** and **p_pentagon** affect Alex's walk home simulation. We'll run experiments with various parameter combinations and visualize the results to understand their impact on:
- Where Alex ends up (Kaia, Pentagon, Railway Station)
- How long the walk takes (in seconds and steps)
- The overall distribution of outcomes

## 1. Import Required Libraries

In [ ]:
import sys
sys.path.insert(0, '../src')

from walk import Location, Walker, Simulation, Experiment
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Set style for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 2. Set Up Simulation Parameters

We'll define the base parameters for our experiments and the ranges of probability values to test.

In [ ]:
# Base simulation parameters
NUM_TRIALS = 1000  # Number of walks per experiment
SEED = 42  # For reproducibility

# Probability ranges to test
p_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# Grid parameters for heatmap
grid_resolution = 11
p_kaia_grid = np.linspace(0.0, 1.0, grid_resolution)
p_pentagon_grid = np.linspace(0.0, 1.0, grid_resolution)

print(f"Running experiments with {NUM_TRIALS} trials each")
print(f"Testing {len(p_values)} different probability values")
print(f"Grid resolution: {grid_resolution}x{grid_resolution} = {grid_resolution**2} combinations")

## 3. Run Simulations with Varying p_kaia

First, we'll vary **p_kaia** from 0.0 to 1.0 while keeping **p_pentagon** constant at 0.5.

In [ ]:
# Vary p_kaia, keep p_pentagon constant
p_pentagon_fixed = 0.5
results_vary_kaia = []

print(f"Running simulations with varying p_kaia (p_pentagon fixed at {p_pentagon_fixed})...")
for p_kaia in p_values:
    # Calculate p_railway
    p_railway = 1.0 - p_kaia - p_pentagon_fixed

    location = Location(p_pentagon=p_pentagon_fixed, p_kaia=p_kaia, p_railway=p_railway)
    experiment = Experiment(location=location, num_trials=NUM_TRIALS, seed=SEED)
    experiment.execute()
    stats = experiment.analyze_results()

    results_vary_kaia.append({
        'p_kaia': p_kaia,
        'p_pentagon': p_pentagon_fixed,
        'p_railway': p_railway,
        'kaia_pct': stats['destinations']['Kaia'],
        'pentagon_pct': stats['destinations']['Pentagon'],
        'railway_pct': stats['destinations']['Railway Station'],
        'avg_seconds': stats['average_seconds'],
        'avg_steps': stats['average_steps']
    })

df_vary_kaia = pd.DataFrame(results_vary_kaia)
print("✓ Complete!")
df_vary_kaia.head()

## 4. Run Simulations with Varying p_pentagon

Now, we'll vary **p_pentagon** from 0.0 to 1.0 while keeping **p_kaia** constant at 0.5.

In [ ]:
# Vary p_pentagon, keep p_kaia constant
p_kaia_fixed = 0.5
results_vary_pentagon = []

print(f"Running simulations with varying p_pentagon (p_kaia fixed at {p_kaia_fixed})...")
for p_pentagon in p_values:
    # Calculate p_railway
    p_railway = 1.0 - p_kaia_fixed - p_pentagon

    location = Location(p_pentagon=p_pentagon, p_kaia=p_kaia_fixed, p_railway=p_railway)
    experiment = Experiment(location=location, num_trials=NUM_TRIALS, seed=SEED)
    experiment.execute()
    stats = experiment.analyze_results()

    results_vary_pentagon.append({
        'p_kaia': p_kaia_fixed,
        'p_pentagon': p_pentagon,
        'p_railway': p_railway,
        'kaia_pct': stats['destinations']['Kaia'],
        'pentagon_pct': stats['destinations']['Pentagon'],
        'railway_pct': stats['destinations']['Railway Station'],
        'avg_seconds': stats['average_seconds'],
        'avg_steps': stats['average_steps']
    })

df_vary_pentagon = pd.DataFrame(results_vary_pentagon)
print("✓ Complete!")
df_vary_pentagon.head()

## 5. Run Simulations with Grid of p_kaia and p_pentagon Values

For a comprehensive view, we'll run simulations for all combinations of p_kaia and p_pentagon (where they sum to ≤ 1.0).

In [ ]:
# Grid search across p_kaia and p_pentagon
results_grid = []

print(f"Running grid search ({grid_resolution}x{grid_resolution} combinations)...")
total_runs = 0
valid_runs = 0

for p_kaia in p_kaia_grid:
    for p_pentagon in p_pentagon_grid:
        total_runs += 1
        # Calculate p_railway
        p_railway = 1.0 - p_kaia - p_pentagon

        # Skip invalid combinations (probabilities must sum to 1.0)
        if p_railway < -0.001:  # Small tolerance for floating point
            continue

        valid_runs += 1
        location = Location(p_pentagon=p_pentagon, p_kaia=p_kaia, p_railway=max(0.0, p_railway))
        experiment = Experiment(location=location, num_trials=NUM_TRIALS, seed=SEED)
        experiment.execute()
        stats = experiment.analyze_results()

        results_grid.append({
            'p_kaia': p_kaia,
            'p_pentagon': p_pentagon,
            'p_railway': max(0.0, p_railway),
            'kaia_pct': stats['destinations']['Kaia'],
            'pentagon_pct': stats['destinations']['Pentagon'],
            'railway_pct': stats['destinations']['Railway Station'],
            'avg_seconds': stats['average_seconds'],
            'avg_steps': stats['average_steps']
        })

df_grid = pd.DataFrame(results_grid)
print(f"✓ Complete! Ran {valid_runs} valid combinations out of {total_runs} total")
print(f"First few results:")
df_grid.head()

## 6. Visualize Results: Impact of p_kaia

Let's visualize how changing p_kaia affects where Alex ends up and how long the walk takes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Destination percentages vs p_kaia
ax1 = axes[0, 0]
ax1.plot(df_vary_kaia['p_kaia'], df_vary_kaia['kaia_pct'], marker='o', label='Kaia', linewidth=2)
ax1.plot(df_vary_kaia['p_kaia'], df_vary_kaia['pentagon_pct'], marker='s', label='Pentagon', linewidth=2)
ax1.plot(df_vary_kaia['p_kaia'], df_vary_kaia['railway_pct'], marker='^', label='Railway', linewidth=2)
ax1.set_xlabel('p_kaia', fontsize=12)
ax1.set_ylabel('Arrival Percentage (%)', fontsize=12)
ax1.set_title('Destination Distribution vs p_kaia\n(p_pentagon = 0.5)', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Average seconds vs p_kaia
ax2 = axes[0, 1]
ax2.plot(df_vary_kaia['p_kaia'], df_vary_kaia['avg_seconds'], marker='o', color='purple', linewidth=2)
ax2.set_xlabel('p_kaia', fontsize=12)
ax2.set_ylabel('Average Time (seconds)', fontsize=12)
ax2.set_title('Average Walk Duration vs p_kaia\n(p_pentagon = 0.5)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Plot 3: Average steps vs p_kaia
ax3 = axes[1, 0]
ax3.plot(df_vary_kaia['p_kaia'], df_vary_kaia['avg_steps'], marker='o', color='green', linewidth=2)
ax3.set_xlabel('p_kaia', fontsize=12)
ax3.set_ylabel('Average Steps', fontsize=12)
ax3.set_title('Average Steps vs p_kaia\n(p_pentagon = 0.5)', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Plot 4: Stacked area chart
ax4 = axes[1, 1]
ax4.fill_between(df_vary_kaia['p_kaia'], 0, df_vary_kaia['kaia_pct'], alpha=0.7, label='Kaia')
ax4.fill_between(df_vary_kaia['p_kaia'], df_vary_kaia['kaia_pct'],
                 df_vary_kaia['kaia_pct'] + df_vary_kaia['pentagon_pct'], alpha=0.7, label='Pentagon')
ax4.fill_between(df_vary_kaia['p_kaia'], df_vary_kaia['kaia_pct'] + df_vary_kaia['pentagon_pct'],
                 100, alpha=0.7, label='Railway')
ax4.set_xlabel('p_kaia', fontsize=12)
ax4.set_ylabel('Cumulative Percentage (%)', fontsize=12)
ax4.set_title('Stacked Destination Distribution vs p_kaia\n(p_pentagon = 0.5)', fontsize=13, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Visualize Results: Impact of p_pentagon

Now let's see how changing p_pentagon affects the simulation outcomes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Destination percentages vs p_pentagon
ax1 = axes[0, 0]
ax1.plot(df_vary_pentagon['p_pentagon'], df_vary_pentagon['kaia_pct'], marker='o', label='Kaia', linewidth=2)
ax1.plot(df_vary_pentagon['p_pentagon'], df_vary_pentagon['pentagon_pct'], marker='s', label='Pentagon', linewidth=2)
ax1.plot(df_vary_pentagon['p_pentagon'], df_vary_pentagon['railway_pct'], marker='^', label='Railway', linewidth=2)
ax1.set_xlabel('p_pentagon', fontsize=12)
ax1.set_ylabel('Arrival Percentage (%)', fontsize=12)
ax1.set_title('Destination Distribution vs p_pentagon\n(p_kaia = 0.5)', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Average seconds vs p_pentagon
ax2 = axes[0, 1]
ax2.plot(df_vary_pentagon['p_pentagon'], df_vary_pentagon['avg_seconds'], marker='o', color='purple', linewidth=2)
ax2.set_xlabel('p_pentagon', fontsize=12)
ax2.set_ylabel('Average Time (seconds)', fontsize=12)
ax2.set_title('Average Walk Duration vs p_pentagon\n(p_kaia = 0.5)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Plot 3: Average steps vs p_pentagon
ax3 = axes[1, 0]
ax3.plot(df_vary_pentagon['p_pentagon'], df_vary_pentagon['avg_steps'], marker='o', color='green', linewidth=2)
ax3.set_xlabel('p_pentagon', fontsize=12)
ax3.set_ylabel('Average Steps', fontsize=12)
ax3.set_title('Average Steps vs p_pentagon\n(p_kaia = 0.5)', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Plot 4: Stacked area chart
ax4 = axes[1, 1]
ax4.fill_between(df_vary_pentagon['p_pentagon'], 0, df_vary_pentagon['kaia_pct'], alpha=0.7, label='Kaia')
ax4.fill_between(df_vary_pentagon['p_pentagon'], df_vary_pentagon['kaia_pct'],
                 df_vary_pentagon['kaia_pct'] + df_vary_pentagon['pentagon_pct'], alpha=0.7, label='Pentagon')
ax4.fill_between(df_vary_pentagon['p_pentagon'], df_vary_pentagon['kaia_pct'] + df_vary_pentagon['pentagon_pct'],
                 100, alpha=0.7, label='Railway')
ax4.set_xlabel('p_pentagon', fontsize=12)
ax4.set_ylabel('Cumulative Percentage (%)', fontsize=12)
ax4.set_title('Stacked Destination Distribution vs p_pentagon\n(p_kaia = 0.5)', fontsize=13, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Create Heatmaps for Parameter Grid

Heatmaps show how both parameters interact to affect simulation outcomes.

In [ ]:
# Create pivot tables for heatmaps
kaia_pivot = df_grid.pivot_table(values='kaia_pct', index='p_pentagon', columns='p_kaia', aggfunc='mean')
pentagon_pivot = df_grid.pivot_table(values='pentagon_pct', index='p_pentagon', columns='p_kaia', aggfunc='mean')
railway_pivot = df_grid.pivot_table(values='railway_pct', index='p_pentagon', columns='p_kaia', aggfunc='mean')
seconds_pivot = df_grid.pivot_table(values='avg_seconds', index='p_pentagon', columns='p_kaia', aggfunc='mean')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Heatmap 1: Kaia arrival percentage
im1 = axes[0, 0].imshow(kaia_pivot, cmap='YlOrRd', aspect='auto', origin='lower')
axes[0, 0].set_xlabel('p_kaia', fontsize=12)
axes[0, 0].set_ylabel('p_pentagon', fontsize=12)
axes[0, 0].set_title('Arrival at Kaia (%)', fontsize=13, fontweight='bold')
plt.colorbar(im1, ax=axes[0, 0], label='Percentage')
axes[0, 0].set_xticks(np.arange(len(p_kaia_grid))[::2])
axes[0, 0].set_xticklabels([f'{p:.1f}' for p in p_kaia_grid[::2]])
axes[0, 0].set_yticks(np.arange(len(p_pentagon_grid))[::2])
axes[0, 0].set_yticklabels([f'{p:.1f}' for p in p_pentagon_grid[::2]])

# Heatmap 2: Pentagon arrival percentage
im2 = axes[0, 1].imshow(pentagon_pivot, cmap='YlGnBu', aspect='auto', origin='lower')
axes[0, 1].set_xlabel('p_kaia', fontsize=12)
axes[0, 1].set_ylabel('p_pentagon', fontsize=12)
axes[0, 1].set_title('Arrival at Pentagon (%)', fontsize=13, fontweight='bold')
plt.colorbar(im2, ax=axes[0, 1], label='Percentage')
axes[0, 1].set_xticks(np.arange(len(p_kaia_grid))[::2])
axes[0, 1].set_xticklabels([f'{p:.1f}' for p in p_kaia_grid[::2]])
axes[0, 1].set_yticks(np.arange(len(p_pentagon_grid))[::2])
axes[0, 1].set_yticklabels([f'{p:.1f}' for p in p_pentagon_grid[::2]])

# Heatmap 3: Railway arrival percentage
im3 = axes[1, 0].imshow(railway_pivot, cmap='Greens', aspect='auto', origin='lower')
axes[1, 0].set_xlabel('p_kaia', fontsize=12)
axes[1, 0].set_ylabel('p_pentagon', fontsize=12)
axes[1, 0].set_title('Arrival at Railway Station (%)', fontsize=13, fontweight='bold')
plt.colorbar(im3, ax=axes[1, 0], label='Percentage')
axes[1, 0].set_xticks(np.arange(len(p_kaia_grid))[::2])
axes[1, 0].set_xticklabels([f'{p:.1f}' for p in p_kaia_grid[::2]])
axes[1, 0].set_yticks(np.arange(len(p_pentagon_grid))[::2])
axes[1, 0].set_yticklabels([f'{p:.1f}' for p in p_pentagon_grid[::2]])

# Heatmap 4: Average walk duration
im4 = axes[1, 1].imshow(seconds_pivot, cmap='viridis', aspect='auto', origin='lower')
axes[1, 1].set_xlabel('p_kaia', fontsize=12)
axes[1, 1].set_ylabel('p_pentagon', fontsize=12)
axes[1, 1].set_title('Average Walk Duration (seconds)', fontsize=13, fontweight='bold')
plt.colorbar(im4, ax=axes[1, 1], label='Seconds')
axes[1, 1].set_xticks(np.arange(len(p_kaia_grid))[::2])
axes[1, 1].set_xticklabels([f'{p:.1f}' for p in p_kaia_grid[::2]])
axes[1, 1].set_yticks(np.arange(len(p_pentagon_grid))[::2])
axes[1, 1].set_yticklabels([f'{p:.1f}' for p in p_pentagon_grid[::2]])

plt.tight_layout()
plt.show()

## 9. Analyze and Compare Results

Let's examine the key findings from our parameter exploration.

In [ ]:
# Find extreme cases
print("=" * 70)
print("KEY FINDINGS FROM PARAMETER EXPLORATION")
print("=" * 70)

# Most likely to reach each destination
idx_max_kaia = df_grid['kaia_pct'].idxmax()
idx_max_pentagon = df_grid['pentagon_pct'].idxmax()
idx_max_railway = df_grid['railway_pct'].idxmax()

print("\n1. DESTINATION PREFERENCES")
print("-" * 70)
print(f"Maximum Kaia arrivals: {df_grid.loc[idx_max_kaia, 'kaia_pct']:.1f}%")
print(f"   with p_kaia={df_grid.loc[idx_max_kaia, 'p_kaia']:.2f}, "
      f"p_pentagon={df_grid.loc[idx_max_kaia, 'p_pentagon']:.2f}")

print(f"\nMaximum Pentagon arrivals: {df_grid.loc[idx_max_pentagon, 'pentagon_pct']:.1f}%")
print(f"   with p_kaia={df_grid.loc[idx_max_pentagon, 'p_kaia']:.2f}, "
      f"p_pentagon={df_grid.loc[idx_max_pentagon, 'p_pentagon']:.2f}")

print(f"\nMaximum Railway arrivals: {df_grid.loc[idx_max_railway, 'railway_pct']:.1f}%")
print(f"   with p_kaia={df_grid.loc[idx_max_railway, 'p_kaia']:.2f}, "
      f"p_pentagon={df_grid.loc[idx_max_railway, 'p_pentagon']:.2f}")

# Walk duration analysis
idx_min_time = df_grid['avg_seconds'].idxmin()
idx_max_time = df_grid['avg_seconds'].idxmax()

print("\n2. WALK DURATION")
print("-" * 70)
print(f"Shortest average walk: {df_grid.loc[idx_min_time, 'avg_seconds']:.1f} seconds")
print(f"   with p_kaia={df_grid.loc[idx_min_time, 'p_kaia']:.2f}, "
      f"p_pentagon={df_grid.loc[idx_min_time, 'p_pentagon']:.2f}")

print(f"\nLongest average walk: {df_grid.loc[idx_max_time, 'avg_seconds']:.1f} seconds")
print(f"   with p_kaia={df_grid.loc[idx_max_time, 'p_kaia']:.2f}, "
      f"p_pentagon={df_grid.loc[idx_max_time, 'p_pentagon']:.2f}")

# Equal probability case
equal_prob_case = df_grid[(df_grid['p_kaia'] >= 0.32) & (df_grid['p_kaia'] <= 0.34) &
                          (df_grid['p_pentagon'] >= 0.32) & (df_grid['p_pentagon'] <= 0.34)]

if not equal_prob_case.empty:
    print("\n3. EQUAL PROBABILITY CASE (≈33% each)")
    print("-" * 70)
    row = equal_prob_case.iloc[0]
    print(f"p_kaia={row['p_kaia']:.2f}, p_pentagon={row['p_pentagon']:.2f}, p_railway={row['p_railway']:.2f}")
    print(f"Results: Kaia={row['kaia_pct']:.1f}%, Pentagon={row['pentagon_pct']:.1f}%, Railway={row['railway_pct']:.1f}%")
    print(f"Average duration: {row['avg_seconds']:.1f} seconds, {row['avg_steps']:.1f} steps")

print("\n" + "=" * 70)

## 10. Observations and Conclusions

Based on the visualizations and analysis above, here are the key observations:

### 📊 **Impact of p_kaia:**
- **Direct correlation**: As p_kaia increases, the probability of Alex ending up at Kaia increases proportionally
- **Trade-off effect**: Higher p_kaia means lower p_railway (since p_pentagon is fixed), so Railway arrivals decrease
- **Walk duration**: Generally increases slightly with higher p_kaia, as Alex is more likely to end up further from AudMax

### 📊 **Impact of p_pentagon:**
- **Direct correlation**: As p_pentagon increases, Pentagon arrivals increase linearly
- **Symmetric trade-off**: Similar to p_kaia, higher p_pentagon reduces Railway arrivals
- **Walk duration**: Shows similar patterns to p_kaia variations

### 🎯 **Interaction Effects (from heatmaps):**
- **Complementary probabilities**: p_kaia and p_pentagon compete for probability mass, with p_railway absorbing the remainder
- **Diagonal patterns**: When p_kaia + p_pentagon ≈ 1.0, Railway arrivals approach 0%
- **Corner cases**:
  - (p_kaia=1.0, p_pentagon=0.0): Alex almost always ends at Kaia
  - (p_kaia=0.0, p_pentagon=1.0): Alex almost always ends at Pentagon
  - (p_kaia=0.0, p_pentagon=0.0): Alex almost always ends at Railway Station

### ⏱️ **Walk Duration Insights:**
- **Longest walks**: Occur when probabilities favor destinations further from AudMax (Kaia and Pentagon)
- **Shortest walks**: Occur with balanced probabilities, suggesting Alex reaches an endpoint more quickly
- **Range**: Average walk times vary from ~100 seconds to ~200+ seconds depending on parameter settings

### 🎲 **Randomness and Predictability:**
- Even with extreme probability settings (e.g., p_kaia=1.0), there's still some variability due to the random walk nature
- The 20% step probability and 50/50 direction choice ensure that outcomes aren't completely deterministic
- Results show good agreement with theoretical expectations (probabilities ≈ outcomes)

### 💡 **Practical Implications:**
If we wanted to control where Alex is most likely to end up:
- **To reach Kaia**: Set high p_kaia (0.7-1.0) and low p_pentagon
- **To reach Pentagon**: Set high p_pentagon (0.7-1.0) and low p_kaia
- **To reach Railway**: Set both p_kaia and p_pentagon to low values (0.0-0.3)
- **For uncertainty**: Use balanced probabilities (≈0.33 each) for maximum unpredictability